
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Bonus Lesson</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Build a Declarative Pipeline with Spark Declarative Pipelines</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Rebuild the Bronze → Silver → Gold pipeline declaratively: define what you want, and let Databricks figure out how to run it.</div>
</div>

</div>


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="padding: 18px 24px; background: #E3F2FD; border: 3px solid #4299E0; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px; font-size: 16pt;">This notebook works differently</div>
    <p>Unlike the previous lessons, you <strong>cannot run these SQL cells interactively</strong>. This notebook is designed to be executed by the <strong>Spark Declarative Pipelines</strong> engine.</p>
    <p>You will:</p>
    <ol style="padding-left: 20px; margin: 8px 0;">
      <li><strong>Read</strong> through the lesson content and SQL definitions below</li>
      <li><strong>Create</strong> an ETL Pipeline in the Jobs &amp; Pipelines UI, pointing at this notebook</li>
      <li><strong>Start</strong> the pipeline and watch it build your tables automatically</li>
      <li><strong>Explore</strong> the results in the Practice notebook</li>
    </ol>
  </div>
</div>
</div>

**In this lesson:** In Lesson 14, you automated the Bronze → Silver → Gold pipeline using a Lakeflow Job with two task notebooks. It worked, but you had to write the orchestration yourself: create the empty tables, run COPY INTO, define the transformations, manage task dependencies.

What if you could just *describe* the tables you want, and let Databricks handle the rest? Let's find out!


<!-- LEARN: Imperative vs Declarative -->
<!-- Template: 2-card-colored-header-guidance -->

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Imperative vs. Declarative Pipelines</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Two approaches to building the same pipeline — one tells Databricks <em>how</em>, the other tells Databricks <em>what</em>.</div>

<div style="display: flex; gap: 20px; justify-content: center;">

<!-- Card 1: Imperative -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #90A5B1; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">Imperative (Lessons 12 & 14)</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      You wrote step-by-step instructions: create an empty table, run COPY INTO, create the Silver table from Bronze, create a temp view, insert into Gold. You managed the order and dependencies yourself.
    </div>
    <div style="background: rgba(144,165,177,0.15); border-left: 4px solid #90A5B1; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>You said:</strong> "First do this, then do that, then do the other thing."
    </div>
  </div>
</div>

<!-- Card 2: Declarative -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #4299E0; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">Declarative (This Lesson)</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      You describe what each table should contain and where the data comes from. Databricks determines the execution order, handles incremental loading, and manages infrastructure automatically.
    </div>
    <div style="background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>You say:</strong> "I want these tables with this data. You figure out how."
    </div>
  </div>
</div>

</div>

<!-- Key point callout -->
<div style="margin-top: 20px; padding: 16px 20px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <strong>Same pipeline, less code, less orchestration.</strong> Spark Declarative Pipelines (SDP) handles execution order, incremental processing, error recovery, and infrastructure; you just define the tables.
  </div>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**What Spark Declarative Pipelines handles for you**

- **Execution order:** SDP reads your table definitions, figures out the dependencies (Bronze feeds Silver, Silver feeds Gold), and runs them in the right order. No task dependencies to configure.
- **Incremental processing:** Streaming tables automatically track which data has been processed. New files get picked up; old files are skipped. This is similar to COPY INTO but fully managed.
- **Infrastructure:** SDP provisions and manages the compute. You don't choose a cluster or configure serverless — it handles that.
- **Error handling:** If a step fails, SDP knows which downstream tables are affected and won't update them with stale data.
- **Data quality:** You can add expectations (quality rules) directly to your table definitions. SDP tracks violations and can warn or drop bad records.

**Two key object types**

- **Streaming tables** are for incremental ingestion. They process new data as it arrives and track what's already been loaded. Use these for your Bronze layer.
- **Materialized views** are for transformations. They recompute from their source each time the pipeline runs. Use these for Silver and Gold layers.

</details>


<!-- LEARN: Three building blocks -->
<!-- Template: vertical-layered-stack (3 layers) -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">The Declarative Pipeline: Three Definitions, One Pipeline</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Each SQL cell below defines one layer of the Medallion Architecture. SDP reads all three and builds the full pipeline automatically.</div>

<div style="display: flex; align-items: stretch; gap: 16px;">

<!-- Left arrow label -->
<div style="
    writing-mode: vertical-lr;
    transform: rotate(180deg);
    text-align: center;
    font-weight: 700;
    font-size: 14pt;
    color: #618794;
    padding: 0 6px;
    display: flex;
    justify-content: flex-end;
">
&larr; RAW TO REFINED
</div>

<!-- Stacked layers -->
<div style="flex: 1; display: flex; flex-direction: column; gap: 6px;">

<!-- Bronze -->
<div style="background: #CD7F32; color: white; border-radius: 8px 8px 4px 4px; padding: 22px 24px; text-align: center;">
  <div style="font-size: 18pt; font-weight: 700;">Streaming Table — Bronze</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;"><code style="background: rgba(255,255,255,0.2); padding: 2px 6px; border-radius: 4px;">CREATE OR REFRESH STREAMING TABLE</code> — ingests new files incrementally</div>
</div>

<!-- Silver -->
<div style="background: #90A5B1; color: white; border-radius: 4px; padding: 18px 24px; text-align: center;">
  <div style="font-size: 16pt; font-weight: 700;">Materialized View — Silver</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.9;"><code style="background: rgba(255,255,255,0.2); padding: 2px 6px; border-radius: 4px;">CREATE OR REFRESH MATERIALIZED VIEW</code> — transforms and validates with expectations</div>
</div>

<!-- Gold -->
<div style="background: #FFAB00; color: #0b2026; border-radius: 4px 4px 8px 8px; padding: 18px 24px; text-align: center;">
  <div style="font-size: 16pt; font-weight: 700;">Materialized View — Gold</div>
  <div style="font-size: 14pt; margin-top: 6px; opacity: 0.85;"><code style="background: rgba(0,0,0,0.08); padding: 2px 6px; border-radius: 4px;">CREATE OR REFRESH MATERIALIZED VIEW</code> — aggregates for business consumption</div>
</div>

</div>

</div>

</div>

---
## Pipeline Definition: Bronze Layer

The streaming table below reads all CSV files from the volume. The `STREAM` keyword tells SDP to track which files have been processed, just like COPY INTO, but fully managed.

Compare this to Lesson 12 where you ran `CREATE TABLE` + `COPY INTO` as two separate steps.

In [0]:
CREATE OR REFRESH STREAMING TABLE current_employees_bronze_sdp
COMMENT "Raw employee data ingested from CSV files in the myfiles volume."
AS SELECT *
FROM STREAM read_files(
  '/Volumes/${my_catalog}/${my_schema}/myfiles/',
  format => 'csv',
  header => true,
  inferSchema => true
);

---
## Pipeline Definition: Silver Layer

The materialized view below transforms Bronze into Silver, which are the same transformations done in Lesson 12 (uppercase Role, add timestamps), but with a bonus: **data quality expectations**.

The `CONSTRAINT` lines define rules that SDP will track:
- `valid_id` — every row must have a non-null ID
- `valid_name` — every row must have a non-null FirstName

If a row violates a `WARN` expectation, it's still included in the table, but the violation is logged. This gives you visibility into data quality without dropping records.

**NOTE:** In SDP, `WARN` is not supported—EXPECT already logs violations and continues, so `EXPECT (...) ON VIOLATION WARN` is equivalent to `EXPECT (...)`.

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW current_employees_silver_sdp (
  CONSTRAINT valid_id EXPECT (ID IS NOT NULL),
  CONSTRAINT valid_name EXPECT (FirstName IS NOT NULL) 
)
COMMENT "Cleaned and enriched employee data with data quality expectations."
AS SELECT
  ID,
  FirstName,
  Country,
  UPPER(Role) AS Role,
  current_timestamp() AS processed_timestamp,
  current_date() AS processed_date
FROM current_employees_bronze_sdp;

---
## Pipeline Definition: Gold Layer

The Gold materialized view aggregates Silver into a business-ready summary, same as the Gold layer in Lesson 12, but in a single declaration instead of a temp view + CREATE TABLE + INSERT OVERWRITE.

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW total_roles_gold_sdp
COMMENT "Employee count by role — business-ready aggregation."
AS SELECT
  Role,
  COUNT(*) AS TotalEmployees
FROM current_employees_silver_sdp
GROUP BY Role;

---
## That's the entire pipeline.

Three SQL statements. No `CREATE TABLE` + `COPY INTO`. No task dependencies. No job configuration. SDP reads these three definitions, determines the order (Bronze → Silver → Gold), and runs them.

Now let's set it up.


<!-- LEARN: Comparison -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 16px;">Side-by-Side: Manual vs. Declarative</div>

<table style="width: 100%; border-collapse: collapse; font-size: 14pt;">
  <tr style="background: #1B5162; color: white;">
    <th style="padding: 12px 16px; text-align: left; border-radius: 8px 0 0 0;">Step</th>
    <th style="padding: 12px 16px; text-align: left;">Imperative (Lessons 12 & 14)</th>
    <th style="padding: 12px 16px; text-align: left; border-radius: 0 8px 0 0;">Declarative (This Lesson)</th>
  </tr>
  <tr style="background: #F9F7F4;">
    <td style="padding: 10px 16px; font-weight: 600;">Bronze</td>
    <td style="padding: 10px 16px;">CREATE TABLE + COPY INTO (2 steps)</td>
    <td style="padding: 10px 16px;">CREATE OR REFRESH STREAMING TABLE (1 step)</td>
  </tr>
  <tr>
    <td style="padding: 10px 16px; font-weight: 600;">Silver</td>
    <td style="padding: 10px 16px;">CREATE OR REPLACE TABLE ... AS SELECT</td>
    <td style="padding: 10px 16px;">CREATE OR REFRESH MATERIALIZED VIEW + expectations</td>
  </tr>
  <tr style="background: #F9F7F4;">
    <td style="padding: 10px 16px; font-weight: 600;">Gold</td>
    <td style="padding: 10px 16px;">Temp view + CREATE TABLE + INSERT OVERWRITE (3 steps)</td>
    <td style="padding: 10px 16px;">CREATE OR REFRESH MATERIALIZED VIEW (1 step)</td>
  </tr>
  <tr>
    <td style="padding: 10px 16px; font-weight: 600;">Orchestration</td>
    <td style="padding: 10px 16px;">Lakeflow Job with 2 tasks + dependencies</td>
    <td style="padding: 10px 16px;">Automatic — SDP determines the order</td>
  </tr>
  <tr style="background: #F9F7F4;">
    <td style="padding: 10px 16px; font-weight: 600;">Data quality</td>
    <td style="padding: 10px 16px;">Not built in — you'd write checks manually</td>
    <td style="padding: 10px 16px;">CONSTRAINT expectations tracked automatically</td>
  </tr>
</table>

</div>

---
## Demo: Create and Run the Pipeline

Now you'll configure an ETL Pipeline in the Databricks UI that uses this notebook as its source.

**Follow these steps:**

1. In the left sidebar, click **Jobs & Pipelines** (right-click → open in a new tab so you can refer back to these instructions)
2. Click **Create** and select **ETL Pipeline**
3. Give your pipeline a name (e.g., `yourname-sdp-bronze-silver-gold`)
4. Select **Edit the catalog and schema**:
   - **Catalog:** `labuser`
   - **Schema:** Select your **get_started_de** schema
5. Click on **Add existing assets**
6. Under **Pipeline root folder**, click **Browse** and select your project folder
7. Under **Source code paths**, click **Browse** and navigate to this notebook:
   - Find your project folder → select **16 Demo (Bonus) - Build a Declarative Pipeline with Spark Declarative Pipelines**
8. Click **ADD**
9. Select **Add configuration**
   - `my_catalog`: `labuser`
   - `my_schema`: Select your **get_started_de** schema
10. Under **Compute**, confirm **Serverless** is selected
11. Click **Create**

### Start the pipeline

1. Click **Start** in the top right of the pipeline editor
2. Watch the pipeline graph appear. You should see three nodes: `current_employees_bronze_sdp` → `current_employees_silver_sdp` → `total_roles_gold_sdp`
3. Each node will progress through: **Queued** → **Running** → **Completed**
4. The pipeline typically takes 2-5 minutes to complete

While it runs, notice how SDP automatically determined the execution order from your SQL definitions. You didn't configure any dependencies, it figured out that Silver reads from Bronze, and Gold reads from Silver.

### Explore the pipeline results

Once the pipeline shows **Completed**:

1. **Click on `current_employees_silver_sdp`** in the pipeline graph
   - Look for the **Data Quality** section. You should see your two expectations (`valid_id` and `valid_name`) with pass/fail counts
2. **Click on any node** and select **View in Catalog Explorer** to see the table details, lineage, and permissions
3. Notice the **lineage** is automatically tracked. SDP knows exactly how each table was built

When you're ready, head to the **Practice** notebook to query and compare the results.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Defined a full Bronze → Silver → Gold pipeline in <strong>3 SQL statements</strong></li>
      <li>Used a <strong>streaming table</strong> for incremental ingestion (Bronze)</li>
      <li>Used <strong>materialized views</strong> for transformations (Silver) and aggregations (Gold)</li>
      <li>Added <strong>data quality expectations</strong> to catch issues automatically</li>
      <li>Configured and ran an <strong>ETL Pipeline</strong> that handled orchestration, compute, and execution order for you</li>
    </ul>
    <div style="margin-top: 12px;">This is the same pipeline you built across Lessons 6 and 7, but declarative instead of imperative. In production, most Databricks data engineers use Spark Declarative Pipelines for exactly this reason: less code, automatic orchestration, and built-in data quality.</div>
  </div>
</div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>